# Imports

In [ ]:
# ============================================================
# ANOM.0) Imports
# ============================================================

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

# Project helpers (you already have these modules)
from src.bench.guardrails_artifacts import (
    build_guardrail_fn_registry,
    load_guardrail_spec,
)

# Optional (recommended): your parquet reader helper from the project
# If read_pq exists in your notebook utils, import it instead of redefining.
read_pq = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 120)

print("Imports OK.")

# Paths + load V3 artifacts (spec + thresholds)

In [ ]:
# ============================================================
# ANOM.1) Load V3 artifacts (spec + thresholds)
# ============================================================

GUARDRAIL_SPEC_PATH = Path("artifacts/guardrails/v3/guardrail_v3_spec.json")
THRESHOLDS_PATH     = Path("artifacts/guardrails/v3/thresholds_v2.json")

assert GUARDRAIL_SPEC_PATH.exists(), f"Missing: {GUARDRAIL_SPEC_PATH}"
assert THRESHOLDS_PATH.exists(), f"Missing: {THRESHOLDS_PATH}"

registry = build_guardrail_fn_registry()

loaded = load_guardrail_spec(
    spec_path=GUARDRAIL_SPEC_PATH,
    thresholds_path=THRESHOLDS_PATH,
    fn_registry=registry,
)

guardrail_v3_rehydrated = loaded["guardrail"]
thresholds_v2_loaded = loaded["thresholds"]

EPS = float(thresholds_v2_loaded.get("epsilon", 1e-9))

print("Loaded guardrail:", guardrail_v3_rehydrated.get("name"), "| components:", len(guardrail_v3_rehydrated["components"]))
print("Threshold keys:", len(thresholds_v2_loaded))
print("EPS:", EPS)

# Load eval universe

In [ ]:
# ============================================================
# ANOM.2) Load anomaly universe (D/G + V3 scored frame)
# Preferred: load from a parquet you produced in the guardrails notebook
# ============================================================

# Option 1 (recommended): point to an explicit parquet file you saved.
# Update this path once, then keep the notebook stable.
EVAL_SCORED_DG_V3_PATH = Path("artifacts/eval_universe/eval_scored_DG_V3.parquet")

if EVAL_SCORED_DG_V3_PATH.exists():
    eval_scored_DG_V3 = pd.read_parquet(EVAL_SCORED_DG_V3_PATH)
    print("Loaded:", EVAL_SCORED_DG_V3_PATH)
else:
    # Option 2: load from your failure-analysis parquet bundle if that's your current storage.
    # This assumes PARQUET_DIR points to the bundle you used in EVAL.1.
    # If you already have read_pq in this repo, we can use it; otherwise fallback to pd.read_parquet.
    PARQUET_DIR = Path("artifacts/failure_analysis_v2/parquet_rich_v2_20260309_093101")  # update if needed
    candidate = PARQUET_DIR / "failure_df_full_v2.parquet"
    assert candidate.exists(), f"Could not find eval universe parquet at {candidate} or {EVAL_SCORED_DG_V3_PATH}"

    eval_scored_DG_V3 = pd.read_parquet(candidate)
    print("Loaded:", candidate)
    print("NOTE: This is failure_df_full_v2. Confirm it reflects D/G+V3 expected_cost in this bundle.")

print("Rows:", len(eval_scored_DG_V3))
print("Cols:", eval_scored_DG_V3.shape[1])
display(eval_scored_DG_V3.head(3))

# Minimal schema sanity (so later cells fail fast)

In [ ]:
# ============================================================
# ANOM.3) Minimal schema sanity checks (fail fast)
# ============================================================

REQ = [
    "row_id", "Rndrng_NPI", "HCPCS_Cd", "Year",
    "observed_cost", "expected_cost", "residual", "abs_residual", "oe_ratio",
    "services", "benes", "expected_cost_support_tier", "has_lag",
    "rbcs_family_desc", "state",
]

missing = [c for c in REQ if c not in eval_scored_DG_V3.columns]
assert not missing, f"eval_scored_DG_V3 is missing required columns: {missing}"

# Numeric coercions (defensive)
for c in ["observed_cost","expected_cost","residual","abs_residual","oe_ratio","services","benes"]:
    eval_scored_DG_V3[c] = pd.to_numeric(eval_scored_DG_V3[c], errors="coerce")

eval_scored_DG_V3["has_lag"] = eval_scored_DG_V3["has_lag"].astype(bool)

# Basic finiteness checks
assert eval_scored_DG_V3["observed_cost"].notna().all(), "observed_cost has NaNs"
assert eval_scored_DG_V3["expected_cost"].notna().all(), "expected_cost has NaNs"
assert np.isfinite(eval_scored_DG_V3["oe_ratio"].to_numpy(dtype="float64")).all(), "oe_ratio has non-finite values"

print("Schema sanity OK.")

# ANOM.1 Compute confidence flags (new, auditable) + helper percentiles

In [ ]:
# ============================================================
# ANOM.1) Confidence flags + within-slice percentiles
# ============================================================

import numpy as np
import pandas as pd

df = eval_scored_DG_V3.copy()

# -----------------------------
# 1) Define a new "is_high_conf" flag (auditable, stable)
#    Tune thresholds later after looking at distributions.
# -----------------------------
MIN_SERVICES = 50
MIN_BENES = 20
HIGH_CONF_TIERS = {"high", "medium_high"}

df["is_high_conf"] = (
    df["expected_cost_support_tier"].astype(str).isin(HIGH_CONF_TIERS)
    & (df["services"].fillna(0) >= MIN_SERVICES)
    & (df["benes"].fillna(0) >= MIN_BENES)
)

# Optional: hot-start only (uncomment if you want this stricter definition)
# df["is_high_conf"] = df["is_high_conf"] & df["has_lag"].astype(bool)

# Keep your existing flag too
df["high_confidence_anomaly_candidate"] = df["high_confidence_anomaly_candidate"].astype(bool)

print("High-conf counts:")
display(pd.DataFrame({
    "flag": ["is_high_conf", "high_confidence_anomaly_candidate"],
    "n_true": [int(df["is_high_conf"].sum()), int(df["high_confidence_anomaly_candidate"].sum())],
    "pct_true_%": [float(df["is_high_conf"].mean()*100), float(df["high_confidence_anomaly_candidate"].mean()*100)]
}))

# -----------------------------
# 2) Create within-slice percentiles for oe_ratio and residual
#    Slices: (HCPCS_Cd, Year) as your most comparable unit.
#    If you prefer RBCS family, change slice_cols.
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    # percent rank in [0,1]; stable and easy
    return s.rank(pct=True, method="average")

df["oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["oe_ratio"].transform(_pct_rank)
df["resid_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["residual"].transform(_pct_rank)

# Convenience: top-x% flags
df["is_top_1pct_oe_in_slice"] = df["oe_pct_in_slice"] >= 0.99
df["is_top_1pct_resid_in_slice"] = df["resid_pct_in_slice"] >= 0.99

print("Percentile features created:", ["oe_pct_in_slice", "resid_pct_in_slice"])
display(df[["HCPCS_Cd","Year","oe_ratio","oe_pct_in_slice","residual","resid_pct_in_slice","services","benes","is_high_conf"]].head(5))

# Save back to the notebook namespace
eval_anom = df
print("Defined: eval_anom (copy of eval_scored_DG_V3 + anomaly features)")

# ANOM.2 Row-level “Top anomalies” action list

This is our “what should a stakeholder look at first” table

### ANOM.2) Row-level top anomalies (action list) (uses `oe_ratio` for slice ranking)

In [ ]:
# ============================================================
# ANOM.2) Row-level top anomalies (action list)
# ============================================================

df = eval_anom.copy()

# Scope: focus on over-expected (positive residual or high OE)
# You can tighten/loosen these later.
row_candidates = df[
    (df["oe_ratio"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
].copy()

# A simple, explainable scoring:
# - prioritize high OE percentile and residual percentile
# - boost if high-confidence
row_candidates["anom_score"] = (
    0.6 * row_candidates["oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

# Keep only over-expected direction (optional but usually desired)
row_candidates = row_candidates[(row_candidates["oe_ratio"] > 1.0) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows = (
    row_candidates.sort_values(["anom_score","oe_ratio","residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio",
        "oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
    ]]
)

print("Top anomalies (row-level):", len(top_anomalies_rows))
display(top_anomalies_rows.head(20))

# Save
anom_top_rows = top_anomalies_rows

### Checking the row numbers where `expected_cost` is 0 or ≤1. 

This will tell us whether we should specially handle the cases where `expected_cost` is very small leading to explosion in O/E ratio and creating false-positive anomaly signals. 

In [ ]:
print(f"eval_scored_DG_V3 has {eval_scored_DG_V3.expected_cost.eq(0).sum()} of rows with expected cost of 0 (zero) (out of {eval_scored_DG_V3.shape[0]} total rows)")
print(f"eval_scored_DG_V3 has {top_anomalies_rows.expected_cost.eq(0).sum()} of rows with expected cost of 0 (zero) (out of {top_anomalies_rows.shape[0]} total rows)")

In [ ]:
tolerances = [1, 1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
counts_eval_scored_DG_V3_obs = []
counts_eval_scored_DG_V3_exp = []
counts_row_candidates_obs = []
counts_row_candidates_exp = []

for tol in tolerances:
    counts_eval_scored_DG_V3_obs.append((eval_scored_DG_V3["observed_cost"].abs() < tol).sum())
    counts_eval_scored_DG_V3_exp.append((eval_scored_DG_V3["expected_cost"].abs() < tol).sum())
    counts_row_candidates_obs.append((row_candidates["observed_cost"].abs() < tol).sum())
    counts_row_candidates_exp.append((row_candidates["expected_cost"].abs() < tol).sum())

c = ["exp < 1", "exp < 1e-1", "exp < 1e-2", "exp < 1e-3", "exp < 1e-4", "exp < 1e-5", "exp < 1e-6"]

temp = pd.DataFrame(
    [
        counts_eval_scored_DG_V3_obs,
        counts_eval_scored_DG_V3_exp,
        counts_row_candidates_obs,
        counts_row_candidates_exp
    ],
    columns=c
)

temp.index = ["eval_scored_DG_V3 observed", "eval_scored_DG_V3 expected", "row_candidates observed", "row_candidates expected"]

temp.reset_index().rename(columns={"index":"source"})

> We don't need to create a special flag for `expected_cost == 0` because there are so few of them. However, we should implement using `log_oe_ratio` which is more robust than `oe_ratio` as it would compress the values for both `observed_cost` and `expected_cost` preventing explosive O/E ratios. 

# ANOM.2.a Row-level top anomalies (robust using `log_oe`)

### ANOM.2.a) Row-level top anomalies (action list) (uses ROBUST `log_oe_ratio` for slice ranking)

> Howeer, since we're still using `_pct_rank` (percentile rank) within each slice (i.e., `(HCPCS_Cd, Year)`) and not "magnitude-aware" approach, where the difference between using `oe_ratio` and `log_oe_ratio` would surface, the expected result is the same as "ANOM.2" above.

In [ ]:
# ============================================================
# ANOM.2.a) Row-level top anomalies (ROBUST: uses log_oe)
# Minimal edit of ANOM.2:
#   - uses log_oe percentiles instead of oe_ratio percentiles
#   - filters on log_oe > 0 (equivalent to oe_ratio > 1)
#   - keeps residual > 0
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# -----------------------------
# 0) Build within-slice percentile for log_oe (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

if "log_oe" not in df.columns:
    raise KeyError("Expected column 'log_oe' not found in eval_anom. (It exists in eval_scored_DG_V3 schema you shared.)")

df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# -----------------------------
# 1) Candidate universe (same idea as ANOM.2)
# -----------------------------
row_candidates = df[
    (df["log_oe"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
    & (df["residual"].notna())
].copy()

# Optional denominator hygiene (recommended toggle)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2  # 0.01; small but avoids ultra-tiny denominators driving extremes

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# -----------------------------
# 2) Scoring (log-based)
# -----------------------------
row_candidates["anom_score_robust"] = (
    0.6 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

# -----------------------------
# 3) Directional filter (robust analog of oe_ratio > 1 and residual > 0)
# -----------------------------
# log_oe > 0 <=> oe_ratio > 1.0
LOG_OE_MIN = 0.0

row_candidates = row_candidates[(row_candidates["log_oe"] > LOG_OE_MIN) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows_robust = (
    row_candidates.sort_values(["anom_score_robust", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_robust",
    ]]
)

print("Top anomalies (row-level, robust log_oe):", len(top_anomalies_rows_robust))
display(top_anomalies_rows_robust.head(20))

anom_top_rows_robust = top_anomalies_rows_robust

# ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter)

### ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter) (filters out small slices after transforming with `_pct_rank()`)

> This builds on "ANOM.2.a" adding an extra robustness step with excluding small-size groups. 

In [ ]:
# ============================================================
# ANOM.2.a.1) Row-level top anomalies (ROBUST + size-aware: log_oe + slice_n filter)
# Minimal edit of ANOM.2.a:
#   - add slice_n per (HCPCS_Cd, Year)
#   - filter to slice_n >= MIN_SLICE_N
#   - include slice_n in output
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# -----------------------------
# 0) Build within-slice percentile for log_oe (HCPCS_Cd, Year)
# -----------------------------
slice_cols = ["HCPCS_Cd", "Year"]

def _pct_rank(s: pd.Series) -> pd.Series:
    return s.rank(pct=True, method="average")

if "log_oe" not in df.columns:
    raise KeyError("Expected column 'log_oe' not found in eval_anom.")

df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# NEW: slice size awareness
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

# -----------------------------
# 1) Candidate universe
# -----------------------------
row_candidates = df[
    (df["log_oe"].notna())
    & (df["expected_cost"].notna())
    & (df["observed_cost"].notna())
    & (df["residual"].notna())
].copy()

# NEW: filter out tiny slices
row_candidates = row_candidates[row_candidates["slice_n"] >= MIN_SLICE_N]

# Optional denominator hygiene (recommended toggle)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# -----------------------------
# 2) Scoring (log-based)
# -----------------------------
row_candidates["anom_score_robust"] = (
    0.6 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.4 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.2 * row_candidates["is_high_conf"].astype(int)
)

row_candidates["anom_reason"] = "top_1pct_log_oe_in_slice + high_conf + positive_residual"

row_candidates["slice_key"] = (
    row_candidates["HCPCS_Cd"].astype(str) + "_" + row_candidates["Year"].astype(int).astype(str)
)

# -----------------------------
# 3) Directional filter
# -----------------------------
LOG_OE_MIN = 0.0
row_candidates = row_candidates[(row_candidates["log_oe"] > LOG_OE_MIN) & (row_candidates["residual"] > 0)]

TOP_N = 200

top_anomalies_rows_robust_sizeaware = (
    row_candidates.sort_values(["anom_score_robust", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "slice_n",  # NEW
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_robust",
        "anom_reason",
        "slice_key"
    ]]
)

print("Top anomalies (row-level, robust + size-aware):", len(top_anomalies_rows_robust_sizeaware))
print(f"Applied slice size filter: slice_n >= {MIN_SLICE_N}")
display(top_anomalies_rows_robust_sizeaware.head(20))

anom_top_rows_robust_sizeaware = top_anomalies_rows_robust_sizeaware

# ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters)

### ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters) (computes a magnitude-based scoring so log-transforming `oe_ratio` is meaningful)

> Here we still compute the percentile of `log_oe` within each slice (i.e., within each `(HCPCS_Cd, Year)` silce) but then instead of taking the top ranking 1% from each slice, we 

In [ ]:
# ============================================================
# ANOM.2.b) Row-level top anomalies (magnitude-aware scoring; log_oe matters)
# - keeps slice percentiles (comparability)
# - adds clipped log magnitude (severity)
# - optional expected_cost floor (denominator hygiene)
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions
needed = ["log_oe", "resid_pct_in_slice", "is_high_conf", "expected_cost", "observed_cost", "residual", "HCPCS_Cd", "Year"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.2.b: {missing}. Run ANOM.1 first.")

# If log_oe_pct_in_slice isn't there, compute it (same slice as before)
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# Candidate universe
row_candidates = df[
    df["log_oe"].notna()
    & df["expected_cost"].notna()
    & df["observed_cost"].notna()
    & df["residual"].notna()
].copy()

# Denominator hygiene (recommended)
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2  # 0.01. Tune later based on expected_cost distribution.

if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# Direction filter (over-expected)
row_candidates = row_candidates[(row_candidates["log_oe"] > 0) & (row_candidates["residual"] > 0)]

# Magnitude transform: clip log_oe so outliers do not dominate too hard.
# Interpretation:
#   log_oe = log(oe_ratio)
#   log_oe=0.0 => oe_ratio=1.0
#   log_oe~0.405 => oe_ratio~1.5
#   log_oe~0.693 => oe_ratio~2.0
#   log_oe~1.609 => oe_ratio~5.0
LOG_CLIP_MAX = 2.0   # oe_ratio ~ 7.39. Any bigger gets capped for scoring stability.
log_mag = np.clip(row_candidates["log_oe"].to_numpy(dtype="float64"), 0.0, LOG_CLIP_MAX) / LOG_CLIP_MAX

# Score: mix comparability + severity + confidence
# - 0.45: within-slice extremeness by log percentile
# - 0.35: within-slice extremeness by residual percentile
# - 0.35: magnitude severity by clipped log
# - 0.15: confidence bump
# Note: weights can exceed 1, that is fine. We care about ranking.
row_candidates["anom_score_mag"] = (
    0.45 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.35 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.35 * log_mag
    + 0.15 * row_candidates["is_high_conf"].astype(int)
)

TOP_N = 200

top_anomalies_rows_mag = (
    row_candidates.sort_values(["anom_score_mag", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_mag",
    ]]
)

print("Top anomalies (row-level, magnitude-aware):", len(top_anomalies_rows_mag))
display(top_anomalies_rows_mag.head(20))

anom_top_rows_mag = top_anomalies_rows_mag

# ANOM.2.b.1) Row-level top anomalies (magnitude-aware + size-aware)

In [ ]:
# ============================================================
# ANOM.2.b.1) Row-level top anomalies (magnitude-aware + size-aware)
# Minimal edit of ANOM.2.b:
#   - add slice_n per (HCPCS_Cd, Year)
#   - filter to slice_n >= MIN_SLICE_N
#   - include slice_n in output
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

needed = ["log_oe", "resid_pct_in_slice", "is_high_conf", "expected_cost", "observed_cost", "residual", "HCPCS_Cd", "Year", "row_id"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.2.b.1: {missing}. Run ANOM.1 first.")

# If log_oe_pct_in_slice isn't there, compute it
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

row_candidates = df[
    df["log_oe"].notna()
    & df["expected_cost"].notna()
    & df["observed_cost"].notna()
    & df["residual"].notna()
].copy()

# NEW: filter out tiny slices
row_candidates = row_candidates[row_candidates["slice_n"] >= MIN_SLICE_N]

# Denominator hygiene
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER:
    row_candidates = row_candidates[row_candidates["expected_cost"] >= MIN_EXPECTED]

# Direction filter
row_candidates = row_candidates[(row_candidates["log_oe"] > 0) & (row_candidates["residual"] > 0)]

# Magnitude transform
LOG_CLIP_MAX = 2.0
log_mag = np.clip(row_candidates["log_oe"].to_numpy(dtype="float64"), 0.0, LOG_CLIP_MAX) / LOG_CLIP_MAX

row_candidates["anom_score_mag"] = (
    0.45 * row_candidates["log_oe_pct_in_slice"].fillna(0)
    + 0.35 * row_candidates["resid_pct_in_slice"].fillna(0)
    + 0.35 * log_mag
    + 0.15 * row_candidates["is_high_conf"].astype(int)
)

row_candidates["anom_reason"] = "top_1pct_log_oe_in_slice + high_conf + positive_residual"

row_candidates["slice_key"] = (
    row_candidates["HCPCS_Cd"].astype(str) + "_" + row_candidates["Year"].astype(int).astype(str)
)

TOP_N = 200

top_anomalies_rows_mag_sizeaware = (
    row_candidates.sort_values(["anom_score_mag", "log_oe", "residual"], ascending=[False, False, False])
    .head(TOP_N)
    .loc[:, [
        "row_id","Rndrng_NPI","provider_type","state","zip5",
        "HCPCS_Cd","hcpcs_desc","rbcs_family_desc","Place_Of_Srvc","Year",
        "slice_n",  # NEW
        "services","benes","expected_cost_support_tier","has_lag","route",
        "observed_cost","expected_cost","residual","oe_ratio","log_oe",
        "log_oe_pct_in_slice","resid_pct_in_slice",
        "is_high_conf","high_confidence_anomaly_candidate",
        "any_guardrail_changed" if "any_guardrail_changed" in row_candidates.columns else "guardrail_applied",
        "guardrail_name",
        "anom_score_mag",
        "anom_reason",
        "slice_key"
    ]]
)

print("Top anomalies (row-level, magnitude-aware + size-aware):", len(top_anomalies_rows_mag_sizeaware))
print(f"Applied slice size filter: slice_n >= {MIN_SLICE_N}")
display(top_anomalies_rows_mag_sizeaware.head(20))

anom_top_rows_mag_sizeaware = top_anomalies_rows_mag_sizeaware

# Summary of row-level anomaly analyses ANOM.2. ANOM.2.a, ANOM.2.a.1, ANOM.2.b, and ANOM.2.b.1.

# ✅ Row-level anomaly surfacing variants (ANOM.2 family)

This section documents the five row-level anomaly approaches we implemented, why each exists, what they share, and what makes each one different. The goal is to have a clear, auditable “two-product” workflow:

Primary product (recommended): ANOM.2.a.1 (log-based + slice-size aware)  
Alternative product (contrast / “magnitude-first”): ANOM.2.b.1 (magnitude-aware + slice-size aware)

## 🎯 What “row-level anomalies” mean here

A row is one provider-service-year observation (roughly: NPI × HCPCS × Year, plus context). A row is suspicious in the over-expected direction when:

- observed cost is above expected, and
- the row looks extreme within its peer slice, and
- we have enough support to trust it (confidence rules), and
- (in size-aware variants) the peer slice is large enough to make “top 1%” meaningful.

## ✅ Common building blocks across all variants

### 1) Candidate universe (basic validity)

All approaches start by filtering to rows with valid core fields:

- observed_cost not null
- expected_cost not null
- residual not null (for robust variants)
- OE or log(OE) not null

### 2) Directionality (over-expected only)

Every approach keeps the anomaly direction consistent:

- positive residual (residual > 0)
- and over-expected:
  - non-log versions: oe_ratio > 1
  - log versions: log_oe > 0 (equivalent to oe_ratio > 1)

### 3) Confidence boost

All approaches incorporate “confidence” so the list is actionable, not noise:

- is_high_conf (services + benes + support tier) enters as a score boost
- we also keep high_confidence_anomaly_candidate in the output table as a fast-track flag for review

### 4) Slice-comparable ranking

All variants rely on “comparability slices”:

- slice key is (HCPCS_Cd, Year) by default
- percentiles are computed within each slice to avoid comparing unrelated services

## 🧠 What differs between variants

You have two main design choices:

### A) How do we measure “over-expected extremeness”?

- ANOM.2: uses oe_ratio percentile inside slice (oe_pct_in_slice)
- ANOM.2.a: uses log_oe percentile inside slice (log_oe_pct_in_slice)
- ANOM.2.b: uses a magnitude-aware score so very large log_oe values get extra emphasis (not just rank)

### B) Do we treat tiny slices as trustworthy?

- Non size-aware: top 1% can be misleading if slice_n is small
- Size-aware (.1 variants): compute slice_n and require slice_n >= 50

## 🧾 Summary of each approach

### ANOM.2 (baseline percentile approach using OE)

Builds within-slice percentiles for:

- oe_ratio (oe_pct_in_slice)
- residual (resid_pct_in_slice)

Scores rows using a weighted percentile blend + confidence boost:

`anom_score = 0.6*oe_pct + 0.4*resid_pct + 0.2*is_high_conf`

Filters:

- oe_ratio > 1 and residual > 0

✅ Good: simple and intuitive  
⚠️ Weakness: OE explodes when expected is tiny (even if rare), and OE tails are skewed

### ANOM.2.a (robust percentile approach using log(OE))

Adds within-slice percentile for:

- log_oe (log_oe_pct_in_slice)

Uses percentiles (still rank-based), but with a more stable signal than raw OE:

`anom_score_robust = 0.6*log_oe_pct + 0.4*resid_pct + 0.2*is_high_conf`

Filters:

- log_oe > 0 and residual > 0

Optional denominator hygiene:

- expected_cost >= MIN_EXPECTED (default 1e-2)

✅ Good: keeps the percentile logic but reduces distortion from extreme OE tails  
⚠️ Weakness: still percentile-based, so “top 1%” depends on slice size

### ANOM.2.a.1 (robust + size-aware) ✅ Recommended

This is 2.a plus slice-size awareness.

Computes:

- slice_n = groupby(HCPCS_Cd, Year).size

Adds filter:

- slice_n >= 50

Keeps:

- log_oe_pct_in_slice, residual percentile, confidence boost

✅ Best for actionability: avoids “winner of a tiny slice” problem  
✅ Most defensible: auditable, stable, comparable, and size-aware  
⚠️ Slight tradeoff: we may miss rare codes with small slices, but that’s a feature not a bug

### ANOM.2.b (magnitude-aware scoring)

This variant is intentionally different. It tries to make ranking sensitive to how extreme the anomaly is, not only whether it’s in the top 1%.

Uses:

- log_oe magnitude (and possibly residual magnitude)

Still keeps percentiles as context, but the score is magnitude-driven.  
Output tends to concentrate around codes with many extremely high log_oe rows (like J3490).

✅ Good: produces a “shock list” of the most extreme rows  
⚠️ Weakness: can over-focus on a few codes where expected is systematically low or unstable, even if slice-relative percentile is already maxed out

### ANOM.2.b.1 (magnitude-aware + size-aware) ✅ Alternative product

This is 2.b plus slice-size awareness.

Adds:

- slice_n >= 50

Keeps:

- magnitude-aware scoring focus

✅ Great “alternative product” to contrast with 2.a.1  
✅ More interpretable for stakeholders who care about absolute extremeness  
⚠️ Can still over-index on a single HCPCS family if it dominates magnitude tails

## 📊 Comparison table (granular)

| Variant | Core OE signal used | “Extremeness” definition | Score type | Directional filter | Denominator hygiene | Slice-size awareness | Strengths | Risks / failure modes | Best use |
|---|---|---|---|---|---|---|---|---|---|
| ANOM.2 | oe_ratio | oe_pct_in_slice + resid_pct_in_slice | Percentile-weighted | oe_ratio > 1 and residual > 0 | None | No | Simple, intuitive, quick | OE tails can be distorted when expected is small | First baseline, sanity check |
| ANOM.2.a | log_oe | log_oe_pct_in_slice+ resid_pct_in_slice | Percentile-weighted | log_oe > 0and residual > 0 | Optional expected_cost >= MIN_EXPECTED | No | More stable than OE, still comparable | “Top 1%” still weak for small slices | Main method before size filter |
| ANOM.2.a.1✅ | log_oe | same as 2.a + slice validity | Percentile-weighted | log_oe > 0and residual > 0 | Optional | ✅ slice_n >= 50 | Most defensible, avoids tiny-slice winners | Might drop rare-code anomalies (intentional) | Primary action list |
| ANOM.2.b | log_oe(magnitude) | magnitude-driven (plus residual) | Magnitude-aware | typically log_oe > 0and residual > 0 | Usually paired with MIN_EXPECTED | No | Finds biggest shocks, very “dramatic” | Can over-focus on codes with unstable expected | Alternative shock list |
| ANOM.2.b.1✅ | log_oe(magnitude) | magnitude-driven + slice validity | Magnitude-aware | log_oe > 0and residual > 0 | Optional | ✅ slice_n >= 50 | Shock list that is also defensible | Still can be dominated by a few HCPCS codes | Secondary product |

## 🧩 “Two-product” framing

### ✅ Product 1:  
ANOM.2.a.1 (Primary, robust + size-aware)

- Best for recurring use
- Least likely to waste reviewer time
- More stable across reruns and code distribution changes

### ✅ Product 2:  
ANOM.2.b.1 (Alternative, magnitude-first + size-aware)

- Best for “what are the most extreme dollar shocks?”
- Useful as a contrast view
- Helps us catch “big outliers” that might not rank highest purely by percentile logic

# ANOM.3 Provider-level “who consistently pops” summary

This answers: “Which NPIs keep showing up across codes/years?”

In [ ]:
# ============================================================
# ANOM.3) Provider-level anomaly summary (who consistently pops)
# ============================================================

df = eval_anom.copy()

# Define "anomalous row" using your fast-track + percentile approach
df["is_row_anomalous"] = (
    (df["oe_ratio"] > 1.0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["oe_pct_in_slice"] >= 0.99)
)

provider_summary = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows=("is_row_anomalous","sum"),
        anom_rate_pct=("is_row_anomalous", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_residual=("residual","median"),
        total_services=("services","sum"),
        total_benes=("benes","sum"),
    )
    .reset_index()
)

# Rank: lots of anomalous rows + not just tiny footprint
provider_summary["provider_anom_score"] = (
    provider_summary["n_anom_rows"]
    + 0.25 * provider_summary["n_unique_codes"]
    + 0.25 * provider_summary["n_unique_years"]
)

TOP_N = 200
anom_top_providers = provider_summary.sort_values(
    ["provider_anom_score","n_anom_rows","anom_rate_pct","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers:", len(anom_top_providers))
display(anom_top_providers.head(30))

# Save
anom_top_providers = anom_top_providers

# ANOM.3.a Provider-level anomaly summary (robust provider-level “who consistently pops”) (ROBUST: uses `log_oe`)

In [ ]:
# ============================================================
# ANOM.3.a) Provider-level anomaly summary (ROBUST: uses log_oe)
# Self-contained: computes log_oe_pct_in_slice if missing
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions that must exist already
for col in ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate"]:
    if col not in df.columns:
        raise KeyError(f"Missing required column for ANOM.3.a: {col}.")

# If log_oe_pct_in_slice isn't persisted, compute it now
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]

    def _pct_rank(s: pd.Series) -> pd.Series:
        return s.rank(pct=True, method="average")

    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(_pct_rank)

# Robust anomalous row definition
df["is_row_anomalous_robust"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
)

provider_summary_robust = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows_robust=("is_row_anomalous_robust","sum"),
        anom_rate_pct_robust=("is_row_anomalous_robust", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_log_oe=("log_oe","median"),
        median_residual=("residual","median"),
        total_services=("services","sum"),
        total_benes=("benes","sum"),
    )
    .reset_index()
)

provider_summary_robust["provider_anom_score_robust"] = (
    provider_summary_robust["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust["n_unique_codes"]
    + 0.25 * provider_summary_robust["n_unique_years"]
)

TOP_N = 200
anom_top_providers_robust = provider_summary_robust.sort_values(
    ["provider_anom_score_robust","n_anom_rows_robust","anom_rate_pct_robust","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (robust log_oe):", len(anom_top_providers_robust))
display(anom_top_providers_robust.head(30))

anom_top_providers_robust = anom_top_providers_robust

# ANOM.3.a.1) Provider-level anomaly summary (ROBUST + size-aware: log_oe + slice_n)

In [ ]:
# ============================================================
# ANOM.3.a.1) Provider-level anomaly summary (ROBUST + size-aware: log_oe + slice_n)
# Minimal edit of ANOM.3.a:
#   - add slice_n per (HCPCS_Cd, Year)
#   - require slice_n >= MIN_SLICE_N for anomalous-row definition
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

for col in ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id"]:
    if col not in df.columns:
        raise KeyError(f"Missing required column for ANOM.3.a.1: {col}.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

df["is_row_anomalous_robust_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["slice_n"] >= MIN_SLICE_N)  # NEW
)

provider_summary_robust_sizeaware = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
    .agg(
        n_rows=("row_id","size"),
        n_anom_rows_robust=("is_row_anomalous_robust_sizeaware","sum"),
        anom_rate_pct_robust=("is_row_anomalous_robust_sizeaware", lambda s: float(s.mean()*100)),
        n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
        n_unique_years=("Year", pd.Series.nunique),
        median_oe=("oe_ratio","median"),
        median_log_oe=("log_oe","median"),
        median_residual=("residual","median"),
        total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
        total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
    )
    .reset_index()
)

provider_summary_robust_sizeaware["provider_anom_score_robust"] = (
    provider_summary_robust_sizeaware["n_anom_rows_robust"]
    + 0.25 * provider_summary_robust_sizeaware["n_unique_codes"]
    + 0.25 * provider_summary_robust_sizeaware["n_unique_years"]
)


TOP_N = 200
anom_top_providers_robust_sizeaware = provider_summary_robust_sizeaware.sort_values(
    ["provider_anom_score_robust","n_anom_rows_robust","anom_rate_pct_robust","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (robust + size-aware):", len(anom_top_providers_robust_sizeaware))
print(f"Applied slice size filter in row definition: slice_n >= {MIN_SLICE_N}")
display(anom_top_providers_robust_sizeaware.head(30))

anom_top_providers_robust_sizeaware = anom_top_providers_robust_sizeaware

# ANOM.3.b) Provider-level anomaly summary (magnitude-aware; log_oe severity)

In [ ]:
# ============================================================
# ANOM.3.b) Provider-level anomaly summary (magnitude-aware; log_oe severity)
# - defines is_row_anomalous_mag
# - ranks providers by count + breadth + severity
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

# Preconditions
needed = ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id", "Rndrng_NPI", "provider_type", "state"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.3.b: {missing}. Run ANOM.1 first.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# Optional denominator hygiene for the anomaly definition
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER and "expected_cost" in df.columns:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# Choose a magnitude threshold in a non-arbitrary way:
# - take a high quantile of log_oe among over-expected rows
# - this adapts to your data distribution
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)]
if len(BASE) == 0:
    raise ValueError("No over-expected rows found (log_oe>0 & residual>0). Check inputs.")
LOG_OE_Q = 0.995  # top 0.5% severity threshold; tune if you want broader/narrower
log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))

print(f"log_oe severity cutoff at q={LOG_OE_Q}: {log_oe_severity_cut:.4f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

# Magnitude-aware anomalous row:
# - direction: over-expected
# - confidence: high_conf OR fast-track flag
# - peer extremeness: top 1% within slice by log percentile
# - severity: log_oe above global severity cutoff
df["is_row_anomalous_mag"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
)

# provider_summary_mag = (
#     df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
#     .agg(
#         n_rows=("row_id","size"),
#         n_anom_rows_mag=("is_row_anomalous_mag","sum"),
#         anom_rate_pct_mag=("is_row_anomalous_mag", lambda s: float(s.mean()*100)),
#         n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
#         n_unique_years=("Year", pd.Series.nunique),

#         # Severity summaries among anomalous rows only
#         max_log_oe_mag=("log_oe", lambda s: float(np.nanmax(s))),
#         p95_log_oe_mag=("log_oe", lambda s: float(np.nanpercentile(s, 95))),
#         median_log_oe=("log_oe","median"),

#         total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
#         total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
#     )
#     .reset_index()
# )

def _masked_max_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag"], "log_oe"]
    return float(s.max()) if len(s) else np.nan

def _masked_p95_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag"], "log_oe"]
    return float(np.nanpercentile(s, 95)) if len(s) else np.nan

provider_summary_mag = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag"].mean() * 100),
          "n_unique_codes": g["HCPCS_Cd"].nunique(),
          "n_unique_years": g["Year"].nunique(),
          "max_log_oe_mag": _masked_max_log_oe(g),
          "p95_log_oe_mag": _masked_p95_log_oe(g),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(g["services"].sum()),
          "total_benes": float(g["benes"].sum()),
      }))
      .reset_index()
)


provider_summary_mag = provider_summary_mag.loc[
    provider_summary_mag["n_anom_rows_mag"] > 0
].copy()

# Rank providers by:
# - count of magnitude anomalies
# - breadth across codes/years
# - plus a small severity bump (p95_log_oe_mag)
provider_summary_mag["provider_anom_score_mag"] = (
    provider_summary_mag["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag["n_unique_codes"]
    + 0.25 * provider_summary_mag["n_unique_years"]
    + 0.10 * provider_summary_mag["p95_log_oe_mag"]
)

TOP_N = 200
anom_top_providers_mag = provider_summary_mag.sort_values(
    ["provider_anom_score_mag","n_anom_rows_mag","anom_rate_pct_mag","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (magnitude-aware):", len(anom_top_providers_mag))
display(anom_top_providers_mag.head(30))

anom_top_providers_mag = anom_top_providers_mag

# ANOM.3.b.1) Provider-level anomaly summary (magnitude-aware + size-aware)

In [ ]:
# ============================================================
# ANOM.3.b.1) Provider-level anomaly summary (magnitude-aware + size-aware)
# Minimal edit of ANOM.3.b:
#   - add slice_n per (HCPCS_Cd, Year)
#   - require slice_n >= MIN_SLICE_N for anomalous-row definition
# ============================================================

import numpy as np
import pandas as pd

df = eval_anom.copy()

needed = ["log_oe", "residual", "is_high_conf", "high_confidence_anomaly_candidate", "HCPCS_Cd", "Year", "row_id", "Rndrng_NPI", "provider_type", "state"]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns for ANOM.3.b.1: {missing}.")

# Ensure log_oe_pct_in_slice exists
if "log_oe_pct_in_slice" not in df.columns:
    slice_cols = ["HCPCS_Cd", "Year"]
    df["log_oe_pct_in_slice"] = df.groupby(slice_cols, dropna=False)["log_oe"].transform(
        lambda s: s.rank(pct=True, method="average")
    )

# NEW: slice size awareness
slice_cols = ["HCPCS_Cd", "Year"]
df["slice_n"] = df.groupby(slice_cols, dropna=False)["row_id"].transform("size")
MIN_SLICE_N = 50

# Optional denominator hygiene
USE_MIN_EXPECTED_FILTER = True
MIN_EXPECTED = 1e-2
if USE_MIN_EXPECTED_FILTER and "expected_cost" in df.columns:
    df = df[df["expected_cost"].notna() & (df["expected_cost"] >= MIN_EXPECTED)].copy()

# Severity cutoff computed like in ANOM.3.b
BASE = df[(df["log_oe"] > 0) & (df["residual"] > 0)]
if len(BASE) == 0:
    raise ValueError("No over-expected rows found (log_oe>0 & residual>0). Check inputs.")
LOG_OE_Q = 0.995
log_oe_severity_cut = float(BASE["log_oe"].quantile(LOG_OE_Q))

print(f"log_oe severity cutoff at q={LOG_OE_Q}: {log_oe_severity_cut:.4f} (oe_ratio ~ {np.exp(log_oe_severity_cut):.2f}x)")

df["is_row_anomalous_mag_sizeaware"] = (
    (df["log_oe"] > 0)
    & (df["residual"] > 0)
    & (df["is_high_conf"] | df["high_confidence_anomaly_candidate"])
    & (df["log_oe_pct_in_slice"] >= 0.99)
    & (df["log_oe"] >= log_oe_severity_cut)
    & (df["slice_n"] >= MIN_SLICE_N)  # NEW
)

# provider_summary_mag_sizeaware = (
#     df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
#     .agg(
#         n_rows=("row_id","size"),
#         n_anom_rows_mag=("is_row_anomalous_mag_sizeaware","sum"),
#         anom_rate_pct_mag=("is_row_anomalous_mag_sizeaware", lambda s: float(s.mean()*100)),
#         n_unique_codes=("HCPCS_Cd", pd.Series.nunique),
#         n_unique_years=("Year", pd.Series.nunique),
#         max_log_oe_mag=("log_oe", lambda s: float(np.nanmax(s))),
#         p95_log_oe_mag=("log_oe", lambda s: float(np.nanpercentile(s, 95))),
#         median_log_oe=("log_oe","median"),
#         total_services=("services","sum") if "services" in df.columns else ("row_id","size"),
#         total_benes=("benes","sum") if "benes" in df.columns else ("row_id","size"),
#     )
#     .reset_index()
# )

def _masked_max_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag_sizeaware"], "log_oe"]
    return float(s.max()) if len(s) else np.nan

def _masked_p95_log_oe(g):
    s = g.loc[g["is_row_anomalous_mag_sizeaware"], "log_oe"]
    return float(np.nanpercentile(s, 95)) if len(s) else np.nan

provider_summary_mag_sizeaware = (
    df.groupby(["Rndrng_NPI","provider_type","state"], dropna=False)
      .apply(lambda g: pd.Series({
          "n_rows": len(g),
          "n_anom_rows_mag": int(g["is_row_anomalous_mag_sizeaware"].sum()),
          "anom_rate_pct_mag": float(g["is_row_anomalous_mag_sizeaware"].mean() * 100),
          "n_unique_codes": g["HCPCS_Cd"].nunique(),
          "n_unique_years": g["Year"].nunique(),
          "max_log_oe_mag": _masked_max_log_oe(g),
          "p95_log_oe_mag": _masked_p95_log_oe(g),
          "median_log_oe": float(g["log_oe"].median()),
          "total_services": float(g["services"].sum()),
          "total_benes": float(g["benes"].sum()),
      }))
      .reset_index()
)

provider_summary_mag_sizeaware = provider_summary_mag_sizeaware.loc[
    provider_summary_mag_sizeaware["n_anom_rows_mag"] > 0
].copy()

provider_summary_mag_sizeaware["provider_anom_score_mag"] = (
    provider_summary_mag_sizeaware["n_anom_rows_mag"]
    + 0.25 * provider_summary_mag_sizeaware["n_unique_codes"]
    + 0.25 * provider_summary_mag_sizeaware["n_unique_years"]
    + 0.10 * provider_summary_mag_sizeaware["p95_log_oe_mag"]
)

TOP_N = 200
anom_top_providers_mag_sizeaware = provider_summary_mag_sizeaware.sort_values(
    ["provider_anom_score_mag","n_anom_rows_mag","anom_rate_pct_mag","total_services"],
    ascending=[False, False, False, False]
).head(TOP_N)

print("Top providers (magnitude-aware + size-aware):", len(anom_top_providers_mag_sizeaware))
print(f"Applied slice size filter in row definition: slice_n >= {MIN_SLICE_N}")
display(anom_top_providers_mag_sizeaware.head(30))

anom_top_providers_mag_sizeaware = anom_top_providers_mag_sizeaware

# ✅ Provider-level anomaly surfacing variants (ANOM.3 family)

This section documents the five provider-level anomaly approaches we implemented, why each exists, what they share, and what makes each one different. The goal is to have a clear, auditable “two-product” workflow:
- ✅ Primary product (recommended): ANOM.3.a.1 (log-based + slice-size aware)
- ✅ Alternative product (contrast, “severity-first”): ANOM.3.b.1 (magnitude-aware + slice-size aware)

---

## 🎯 What “provider-level anomalies” mean here

A provider-level anomaly summary answers:

“Which providers repeatedly show up as extreme over-expected relative to comparable peers?”

Key idea: we are not ranking providers by average cost. You are ranking them by the count and breadth of row-level tail events that pass your rules.

A provider “consistently pops” when they have:
- multiple anomalous rows (NPI × HCPCS × Year rows),
- across multiple codes and years,
- in slices where the peer group is large enough to trust the percentile meaning (in size-aware variants).

---

## ✅ Common building blocks across all variants

### 1) Row-level anomaly flag drives everything

Every provider table begins by defining a boolean row flag (one row = one provider-service-year record). That row flag is then aggregated to provider-level counts and rates.

### 2) Directionality: over-expected only

All variants enforce the same “direction”:
- residual > 0 (observed above expected)
- and over-expected signal:
  - OE versions: oe_ratio > 1
  - log versions: log_oe > 0 (equivalent to OE > 1)

### 3) Confidence gating

All variants require rows to be credible using:
- is_high_conf (your auditable rule: support tier + minimum services + minimum benes)
- OR high_confidence_anomaly_candidate (your legacy fast-track flag)

### 4) Slice-comparable extremeness

All variants use the peer slice (HCPCS_Cd, Year) to avoid comparing unlike services.
- percentile computed inside slice, then used as “is this row extreme relative to peers for the same code/year?”

### 5) Provider ranking uses “count + breadth”

Every provider score is built around:
- how many anomalous rows (n_anom_rows_*)
- breadth across codes (n_unique_codes)
- breadth across years (n_unique_years)

---

## 🧠 What differs between variants

You have two main design choices:

### A) What is the “peer extremeness” signal?
- ANOM.3: oe_pct_in_slice (percentile of oe_ratio within slice)
- ANOM.3.a: log_oe_pct_in_slice (percentile of log_oe within slice, same ordering as OE but numerically better behaved)
- ANOM.3.b: adds a global severity cutoff log_oe >= quantile(...) so “how huge is it” matters

### B) Do we trust tiny slices?
- Non size-aware: “top 1%” can be meaningless when slice size is small
- Size-aware (.1 variants): compute slice_n and require slice_n >= 50

---

## 🧾 Summary of each approach

### ANOM.3 (baseline provider tail frequency using OE)

- Defines anomalous row:
  - oe_ratio > 1 and residual > 0
  - (is_high_conf OR high_confidence_anomaly_candidate)
  - oe_pct_in_slice >= 0.99
- Aggregates per provider:
  - counts (n_anom_rows), rates, breadth, medians, totals
- Ranks by:
  - n_anom_rows + 0.25*n_unique_codes + 0.25*n_unique_years

✅ Good: simple, intuitive baseline  
⚠️ Weakness: raw OE can be distorted by denominator issues, and “top 1%” can be misleading in tiny slices

---

### ANOM.3.a (robust provider tail frequency using log(OE))

- Same logic as ANOM.3 but uses:
  - log_oe > 0 instead of oe_ratio > 1
  - log_oe_pct_in_slice >= 0.99 instead of oe_pct_in_slice >= 0.99

✅ Good: same conceptual logic, less prone to OE numeric weirdness  
⚠️ Important: because log is monotonic, the top-1% set per slice usually matches ANOM.3, so provider ranking often stays the same

---

### ANOM.3.a.1 (robust + size-aware) ✅ Recommended primary product

This is ANOM.3.a plus slice-size awareness.
- Adds:
  - slice_n = size(HCPCS_Cd, Year)
  - requires slice_n >= 50 inside the anomalous-row definition
- Aggregates and ranks like ANOM.3.a

✅ Best defensibility: avoids “winner of a tiny peer group” driving provider rank  
✅ Best stability: fewer false positives caused by thin slices  
⚠️ Tradeoff: drops rare-code slices intentionally (feature, not bug)

---

### ANOM.3.b (magnitude-aware provider anomalies; severity-first)

This variant is intentionally different: it emphasizes severity, not just percentile rank.
- Keeps peer extremeness:
  - log_oe_pct_in_slice >= 0.99
- Adds global severity cutoff:
  - log_oe >= quantile(BASE, 0.995) where BASE = over-expected rows
- Provider ranking adds severity bump:
  - + 0.10 * p95_log_oe_mag (severity summary)

✅ Good: produces a “severity-first provider list” for extreme shocks  
⚠️ Weakness: without slice-size filtering it can still admit tiny-slice outliers that look dramatic but are less defensible

---

### ANOM.3.b.1 (magnitude-aware + size-aware) ✅ Recommended alternative product

This is ANOM.3.b plus slice-size awareness (same slice_n >= 50 rule).

✅ Best alternative view: “who has the most severe tail events, in credible peer contexts?”  
✅ More stakeholder-friendly for “big shock” review  
⚠️ Can yield fewer anomalies per provider (often 1–3), so it’s more of a targeted worklist than a “repeat offenders” list

---

## 📊 Comparison table (granular)

| Variant | Core over-expected signal | Peer extremeness definition | Score type | Directional filter | Confidence rule | Denominator hygiene | Slice-size awareness | Severity-aware | Strengths | Risks / failure modes | Best use |
|---|---|---|---|---|---|---|---|---|---|---|---|
| ANOM.3 | oe_ratio | oe_pct_in_slice >= 0.99 | Count + breadth | oe_ratio>1 & residual>0 | is_high_conf OR legacy fast-track | None | No | No | Simple baseline; easy to explain | Tiny slices can dominate; OE can be numerically distorted | Baseline sanity check |
| ANOM.3.a | log_oe | log_oe_pct_in_slice >= 0.99 | Count + breadth | log_oe>0 & residual>0 | same | None | No | No | More numerically stable OE signal | Often yields same set as ANOM.3 (monotonic transform) | “Robust baseline” before size filter |
| ANOM.3.a.1 ✅ | log_oe | same as 3.a + slice validity | Count + breadth | log_oe>0 & residual>0 | same | None | ✅ slice_n>=50 | No | Most defensible; avoids tiny-slice winners; stable | Drops rare small-slice anomalies (intentional) | Primary provider list (“repeat offenders”) |
| ANOM.3.b | log_oe + severity cutoff | log_oe_pct_in_slice>=0.99 + log_oe>=q(.995) | Count + breadth + small severity bump | log_oe>0 & residual>0 | same | Optional expected_cost>=MIN_EXPECTED | No | ✅ Yes | “Shock list” view; severity-first | Tiny slices can still create dramatic but weakly supported shocks | Alternative severity-first review |
| ANOM.3.b.1 ✅ | log_oe + severity cutoff | same as 3.b + slice validity | Count + breadth + severity bump | log_oe>0 & residual>0 | same | Optional | ✅ slice_n>=50 | ✅ Yes | Severity-first but defensible; good worklist | May produce few rows per provider (often 1–3) | Alternative provider product (“big shocks”) |

---

## 🧩 “Two-product” framing (what we now have)

### ✅ Product 1: ANOM.3.a.1 (Primary, robust + size-aware)

- Best for ongoing provider surveillance and “repeat offenders”
- Most defensible to auditors and stakeholders
- Least likely to waste reviewer time on thin-slice artifacts

### ✅ Product 2: ANOM.3.b.1 (Alternative, magnitude-first + size-aware)

- Best for “who has the most severe tail events?”
- Great for targeted deep-dives and escalation review
- Complements the primary list by focusing on severity rather than frequency

---

# Visualizations 

# VIZ.A) "What is an anomaly?" foundation chart (highlight rows that are in primary row list (ANOM.2.a.1))

In [ ]:
# ============================================================
# VIZ.A) "What is an anomaly?" foundation chart
# Scatter: log_oe vs residual
#  - marker/color shows is_high_conf
#  - highlight rows that are in primary row list (ANOM.2.a.1)
#  - reference lines at log_oe=0 and residual=0
# Optional: facet by route (hot_start vs cold_start)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# Inputs (expected to exist)
# -----------------------------
# eval_anom: your full anomaly universe (eval_scored_DG_V3 + features from ANOM.1)
# anom_top_rows_primary: your primary row-level worklist from ANOM.2.a.1
#   - if your variable name differs, set PRIMARY_ROWS_DF below accordingly

PRIMARY_ROWS_DF = None
for _cand in ["anom_top_rows_primary", "anom_top_rows_robust_sizeaware", "anom_top_rows_robust", "anom_top_rows"]:
    if _cand in globals():
        PRIMARY_ROWS_DF = globals()[_cand]
        print(f"Using primary rows from: {_cand}")
        break

if PRIMARY_ROWS_DF is None:
    raise NameError(
        "Could not find primary row list in globals. Expected one of: "
        "anom_top_rows_primary, anom_top_rows_robust_sizeaware, anom_top_rows_robust, anom_top_rows"
    )

if "eval_anom" not in globals():
    raise NameError("Missing eval_anom. Run ANOM.1 first to create eval_anom.")

df = eval_anom.copy()

# -----------------------------
# Preconditions / required cols
# -----------------------------
req_cols = ["row_id", "log_oe", "residual", "is_high_conf"]
missing = [c for c in req_cols if c not in df.columns]
if missing:
    raise KeyError(f"eval_anom is missing required columns for this plot: {missing}")

# Ensure numeric
df["log_oe"] = pd.to_numeric(df["log_oe"], errors="coerce")
df["residual"] = pd.to_numeric(df["residual"], errors="coerce")
df["is_high_conf"] = df["is_high_conf"].astype(bool)

# Identify primary rows
primary_ids = set(pd.to_numeric(PRIMARY_ROWS_DF["row_id"], errors="coerce").dropna().astype(int).tolist())
df["is_primary_row"] = pd.to_numeric(df["row_id"], errors="coerce").fillna(-1).astype(int).isin(primary_ids)

# Keep finite
plot_df = df[np.isfinite(df["log_oe"]) & np.isfinite(df["residual"])].copy()

# Optional: clip extreme residuals for readability (toggle)
CLIP_RESID = False
RESID_PCT = 0.999  # clip at 99.9th percentile of abs residual, if enabled
if CLIP_RESID:
    cap = float(np.nanquantile(np.abs(plot_df["residual"].to_numpy()), RESID_PCT))
    plot_df["residual_plot"] = np.clip(plot_df["residual"], -cap, cap)
else:
    plot_df["residual_plot"] = plot_df["residual"]

# -----------------------------
# Plot helpers
# -----------------------------
def _scatter_panel(ax, d: pd.DataFrame, title: str) -> None:
    # Background: not high confidence
    bg_low = d[(~d["is_high_conf"]) & (~d["is_primary_row"])]
    bg_high = d[(d["is_high_conf"]) & (~d["is_primary_row"])]
    hi = d[d["is_primary_row"]]

    # background points
    ax.scatter(bg_low["log_oe"], bg_low["residual_plot"], s=8, alpha=0.15, marker="o", label="not high-conf")
    ax.scatter(bg_high["log_oe"], bg_high["residual_plot"], s=10, alpha=0.20, marker="^", label="high-conf")

    # highlight primary rows on top
    ax.scatter(hi["log_oe"], hi["residual_plot"], s=35, alpha=0.9, marker="o", label="PRIMARY (ANOM.2.a.1)")

    # reference lines
    ax.axvline(0.0, linewidth=1)
    ax.axhline(0.0, linewidth=1)

    ax.set_title(title)
    ax.set_xlabel("log_oe (log(observed/expected))")
    ax.set_ylabel("residual (observed - expected)")

    # make legend compact
    ax.legend(frameon=False, fontsize=9, loc="best")


# -----------------------------
# Option 1: single chart (no faceting)
# -----------------------------
FACET_BY_ROUTE = True  # set False if you want just one plot

if not FACET_BY_ROUTE:
    fig = plt.figure(figsize=(10, 6))
    ax = fig.add_subplot(111)
    _scatter_panel(ax, plot_df, "What is an anomaly? log_oe vs residual")
    plt.tight_layout()
    plt.show()

# -----------------------------
# Option 2: facet by route (hot_start vs cold_start)
# -----------------------------
else:
    if "route" not in plot_df.columns:
        raise KeyError("FACET_BY_ROUTE=True but eval_anom is missing 'route' column.")

    # Normalize route labels a bit
    plot_df["route"] = plot_df["route"].astype(str)

    routes = ["hot_start", "cold_start"]
    # include any other routes if present
    other_routes = [r for r in sorted(plot_df["route"].dropna().unique()) if r not in routes]
    routes = routes + other_routes

    n = len(routes)
    fig = plt.figure(figsize=(12, 5 * max(1, int(np.ceil(n / 2)))))

    # 2 columns layout
    ncols = 2
    nrows = int(np.ceil(n / ncols))

    for i, r in enumerate(routes, start=1):
        ax = fig.add_subplot(nrows, ncols, i)
        d = plot_df[plot_df["route"] == r]
        _scatter_panel(ax, d, f"Route = {r} (log_oe vs residual)")

    plt.tight_layout()
    plt.show()

In [ ]:
# Look at the smallest positive residual among log_oe > 0 rows
tmp = eval_anom.loc[eval_anom["log_oe"] > 0, ["observed_cost","expected_cost","residual","log_oe"]].copy()
tmp["abs_resid"] = tmp["residual"].abs()
tmp.sort_values("abs_resid").head(20)

In [ ]:
import numpy as np
import pandas as pd

tmp = eval_anom.copy()

# Use the same canonical definitions used in eval_scored_DG_V3
EPS = 1e-9

obs = pd.to_numeric(tmp["observed_cost"], errors="coerce").to_numpy(dtype="float64")
exp = pd.to_numeric(tmp["expected_cost"], errors="coerce").to_numpy(dtype="float64")
res = pd.to_numeric(tmp["residual"], errors="coerce").to_numpy(dtype="float64")
oe  = pd.to_numeric(tmp["oe_ratio"], errors="coerce").to_numpy(dtype="float64")
log = pd.to_numeric(tmp["log_oe"], errors="coerce").to_numpy(dtype="float64")

res_recalc = obs - exp
oe_recalc  = obs / (exp + EPS)          # additive denom, matches your canonicalization
log_recalc = np.log1p(obs) - np.log1p(exp)

print("Rows:", len(tmp))

print("residual mismatches (abs diff > 1e-10):",
      int(np.sum(~np.isclose(res, res_recalc, atol=1e-10, rtol=0.0, equal_nan=True))))

print("oe_ratio mismatches (abs diff > 1e-8):",
      int(np.sum(~np.isclose(oe, oe_recalc, atol=1e-8, rtol=0.0, equal_nan=True))))

print("log_oe mismatches (abs diff > 1e-10):",
      int(np.sum(~np.isclose(log, log_recalc, atol=1e-10, rtol=0.0, equal_nan=True))))

# Show a few worst log_oe diffs if any
diff = log - log_recalc
mask = np.abs(diff) > 1e-10
if mask.any():
    worst_idx = np.argsort(np.abs(diff[mask]))[::-1][:20]
    df_worst = tmp.loc[mask, ["row_id","observed_cost","expected_cost","residual","oe_ratio","log_oe",
                              "guardrail_name","any_guardrail_changed"]].copy()
    df_worst["log_oe_recalc"] = log_recalc[mask]
    df_worst["log_oe_diff"] = diff[mask]
    display(df_worst.iloc[worst_idx])
else:
    print("No material log_oe diffs found.")